In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

# <span style="color:blue;">**Control**</span>

In [3]:
study = "Control"

### 2. Electric Health Record

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [4]:
# file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/parameters of interest.xlsx"
file_path = "/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data/parameters of interest.xlsx"
file_name = pd.ExcelFile(file_path).sheet_names
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes',
 'procedures']

---

# **Extract <span style="color:blue;">Control**</span> 

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

## **STEP 1**. Filter patients (no pathology, BI-RADS = 1, 2) then merge with patients with DBT available
### (OUTPUT) candidate_cohort

enteredit_findings
* Columns: COMPOSITION_NAME, FINDING_CATEGORY, FINDING_REC, EXAM_COMPLETED_DATE
* Cancer: 
    * FINDING_CATEGORY = 1, 2

pathology
* Columns: BX_ID, PATHOLOGY_DATE, LESION_CLASS, SIDE
* Control: 
    * If pathology exists --> exclude

### <span style="color:#FF6347;">**READ**</span> file (EHR: enteredit_findings, pathology)

In [5]:
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes',
 'procedures']

In [6]:
fn = file_name[0]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
birads = pd.read_excel(file_path)

In [7]:
birads.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
0,4330000534,78048053,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-03-28,2
1,4330000534,64787403,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-05-18,2
2,4330000534,63280481,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-06-20,2


In [8]:
fn = file_name[1]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
pathology = pd.read_excel(file_path)

In [9]:
pathology.head(3)

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
0,4330189702,418851,2019-04-26,Malignant,R
1,4330311096,450823,2016-11-10,Malignant,R
2,4330311096,459526,2016-11-28,Malignant,R


### **1. Filter out patients with pathology**

In [10]:
birads.drop(columns="FINDING_LOCATION", inplace=True)

In [11]:
# Identify all patient IDs that have pathology records
pathology_pids = pathology['PATIENT_STUDY_ID'].unique()

# Exclude any patient who appears in pathology at all
birads_no_pathology = birads[~birads['PATIENT_STUDY_ID'].isin(pathology_pids)].copy()

print(f"Patients (BI-RADS) : {birads['PATIENT_STUDY_ID'].nunique()}")
print(f"Patients removed (have pathology) : {birads['PATIENT_STUDY_ID'].nunique() - birads_no_pathology['PATIENT_STUDY_ID'].nunique()}")
print(f"Patients (BI-RADS, no pathology) : {birads_no_pathology['PATIENT_STUDY_ID'].nunique()}")

Patients (BI-RADS) : 38339
Patients removed (have pathology) : 362
Patients (BI-RADS, no pathology) : 37977


### **2. Filter BI-RADS findings for Control criteria** (BI-RADS 1 or 2)

In [12]:
# Keep only BI-RADS 1 (Negative) and BI-RADS 2 (Benign) findings
normal_categories = ['1 - Negative', '2 - Benign finding']

birads_control = birads_no_pathology[
    birads_no_pathology['FINDING_CATEGORY'].isin(normal_categories)
].copy()


birads_control.reset_index(inplace=True, drop=True)

print(f"Rows for BIRADS 1 or 2 filter      : {len(birads_control)}")

Rows for BIRADS 1 or 2 filter      : 234737


In [13]:
# Handle bilateral exams: no SIDE + duplicate_count ≥ 2 → expand to L and R
no_side  = birads_control.copy()

bilateral_mask = no_side['duplicate_count'] >= 2
bilateral      = no_side[bilateral_mask].copy()
rest           = no_side[~bilateral_mask].copy()

bilateral_L = bilateral.copy(); bilateral_L['SIDE'] = 'L'
bilateral_R = bilateral.copy(); bilateral_R['SIDE'] = 'R'

birads_control = pd.concat(
    [bilateral_L, bilateral_R, rest], ignore_index=True
)
print(f"birads_control             : {len(birads_control)}")

birads_control             : 415095


In [30]:
birads_control.sort_values(by=["PATIENT_STUDY_ID", "EXAM_COMPLETED_DATE", "ACCESSION_NUMBER"], inplace=True)
birads_control.reset_index(inplace=True, drop=True)

In [31]:
birads_control

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,SIDE
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,L
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,R
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,L
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,R
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,NaN
...,...,...,...,...,...,...,...,...
415090,4339988199,61697072,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-02-21,2,R
415091,4339988199,67718778,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-02-25,2,L
415092,4339988199,67718778,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-02-25,2,R
415093,4339988199,452549960,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2022-03-01,2,L


In [43]:
df_step1 = birads_control.copy()

### <span style="color:#FF6347;">**SAVE**</span> file

In [44]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "candidate_cohort" + ".xlsx")
df_step1.to_excel(output_file, index=False)

## **STEP 2**. <span style="color:blue;">**Control**</span>: Label  "Index" = <span style="color:#8A2BE2;">**INDEX**</span> or <span style="color:#00BFFF;">**INDEX-1**</span>
### (OUTPUT) candidate_cohort

### <span style="color:#FF6347;">**READ**</span> file (candidate_cohort)

In [84]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "candidate_cohort" + ".xlsx")
control_cohort = pd.read_excel(output_file)

In [85]:
control_cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(control_cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")

In [86]:
control_cohort["PATIENT_STUDY_ID"].nunique()

37977

In [87]:
control_cohort.head(5)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,L,NaN,NaN,NaN
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,R,NaN,NaN,NaN
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,L,NaN,NaN,NaN
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,R,NaN,NaN,NaN
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,NaN,INDEX-2,754.0,24.8


## 1. Locate <span style="color:blue;">**Control**</span> year, "Index" = <span style="color:#8A2BE2;">**INDEX**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [88]:
threshold_group = pd.Timedelta(days=365/12 * 1.5)  # 1.5-month grouping

In [89]:
cohort = control_cohort.copy()
cohort['Index'] = None
cohort['EXAM_COMPLETED_DATE'] = pd.to_datetime(cohort['EXAM_COMPLETED_DATE'])

In [90]:
# ── STEP 1: Locate INDEX ──────────────────────────────────────────────────────
# 1a. INDEX = latest SCREEN exam with BIRADS 1/2 and image available
# idx_screen = cohort['Study'] == 'SCREEN'
idx_normal = cohort['FINDING_CATEGORY'].isin(['1 - Negative', '2 - Benign finding'])

latest_index = (
    cohort[idx_normal]
    .groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE']
    .max()                    # <── latest qualifying exam
    .reset_index()
    .rename(columns={'EXAM_COMPLETED_DATE': 'INDEX_DATE'})
)

cohort = cohort.merge(latest_index, on='PATIENT_STUDY_ID', how='left')

# 1b. Mark exams within 1.5 months before (and including) the anchor as INDEX
mask_index = (
    cohort['INDEX_DATE'].notna() &
    (cohort['EXAM_COMPLETED_DATE'] <= cohort['INDEX_DATE']) &
    (cohort['INDEX_DATE'] - cohort['EXAM_COMPLETED_DATE'] <= threshold_group)
)
cohort.loc[mask_index, 'Index'] = 'INDEX'

print(f"✅ INDEX rows    : {mask_index.sum()}")
print(f"   Patients      : {cohort[mask_index]['PATIENT_STUDY_ID'].nunique()}")

✅ INDEX rows    : 75210
   Patients      : 37977


In [91]:
cohort.head(20)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months,INDEX_DATE
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,L,None,NaN,NaN,2022-06-25
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,R,None,NaN,NaN,2022-06-25
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,L,None,NaN,NaN,2022-06-25
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,R,None,NaN,NaN,2022-06-25
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,NaN,None,754.0,24.8,2022-06-25
5,4330000534,60687102,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-06-10,1,NaN,None,745.0,24.5,2022-06-25
6,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,L,None,387.0,12.7,2022-06-25
7,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,R,None,387.0,12.7,2022-06-25
8,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,L,INDEX,NaN,NaN,2022-06-25
9,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,R,INDEX,NaN,NaN,2022-06-25


## 2. Locate <span style="color:blue;">**Control with 1 year follow-up**</span> year, "Index" = <span style="color:#00BFFF;">**INDEX-1**</span>. Adjust <span style="color:pink;"> **screening interval (OR search period)**</span> 

Add threshold labeling for the same index time point (threshold ~1.5 months)

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [92]:
# ── Parameters ────────────────────────────────────────────────────────────────
min_months      = pd.Timedelta(days=0)              # how far back to look (min)
max_months      = pd.Timedelta(days=365/12 * 30)   # how far back to look (max)
threshold_group = pd.Timedelta(days=365/12 * 1.5)  # 1.5-month grouping

In [93]:
cohort['exam_to_index_diff'] = cohort['INDEX_DATE'] - cohort['EXAM_COMPLETED_DATE']

mask_window = (
    cohort['exam_to_index_diff'].notna() &
    (cohort['exam_to_index_diff'] > min_months) &
    (cohort['exam_to_index_diff'] <= max_months) &
    (cohort['Index'] != 'INDEX') 
)

def assign_index_labels(dates):
    dates_sorted = sorted(set(dates), reverse=True)  # newest → oldest
    labels = {}
    rank = 0
    group_anchor = None
    for d in dates_sorted:
        if group_anchor is None or (group_anchor - d) > threshold_group:
            rank += 1
            group_anchor = d
        labels[d] = f'INDEX-{rank}'
    return labels

window_rows = cohort[mask_window].copy()
date_label_map = {}
for pid, grp in window_rows.groupby('PATIENT_STUDY_ID'):
    mapping = assign_index_labels(grp['EXAM_COMPLETED_DATE'])
    for date, label in mapping.items():
        date_label_map[(pid, date)] = label

cohort.loc[mask_window, 'Index'] = cohort.loc[mask_window].apply(
    lambda row: date_label_map.get(
        (row['PATIENT_STUDY_ID'], row['EXAM_COMPLETED_DATE']),
        row['Index']
    ),
    axis=1
)

print(cohort['Index'].value_counts(dropna=False))

Index
None       219360
INDEX       75210
INDEX-1     70963
INDEX-2     46813
INDEX-3      2502
INDEX-4       204
INDEX-5        40
INDEX-6         3
Name: count, dtype: int64


In [94]:
# ── STEP 3: time_to_INDEX ─────────────────────────────────────────────────────
is_prior = cohort['Index'].str.match(r'^INDEX-\d+$', na=False)

cohort['time_to_INDEX_days'] = (
    cohort['INDEX_DATE'] - cohort['EXAM_COMPLETED_DATE']
).dt.days

cohort.loc[~is_prior, 'time_to_INDEX_days'] = pd.NA
cohort['time_to_INDEX_months'] = (cohort['time_to_INDEX_days'] / 30.44).round(1)

cohort = cohort.drop(columns=['INDEX_DATE', 'exam_to_index_diff'])

In [95]:
cohort.head(20)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,L,None,NaN,NaN
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,R,None,NaN,NaN
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,L,None,NaN,NaN
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,R,None,NaN,NaN
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,NaN,INDEX-2,754.0,24.8
5,4330000534,60687102,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-06-10,1,NaN,INDEX-2,745.0,24.5
6,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,L,INDEX-1,387.0,12.7
7,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,R,INDEX-1,387.0,12.7
8,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,L,INDEX,NaN,NaN
9,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,R,INDEX,NaN,NaN


### <span style="color:#FF6347;">**SAVE**</span> file

In [98]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'candidate_cohort' + ".xlsx")
cohort.to_excel(output_file, index=False)

In [99]:
cohort["PATIENT_STUDY_ID"].nunique()

37977

## **STEP 3**. Extract <span style="color:blue;">**Normal**</span> cohort only
### (Output) normal, normal_cohort

#### <span style="color:#FF6347;">**READ**</span> file (candidate_cohort)

In [100]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'control_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'candidate_cohort' + ".xlsx")
cohort = pd.read_excel(output_file)

In [101]:
cohort.head(20)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months
0,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,L,NaN,NaN,NaN
1,4330000534,78048053,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-03-28,2,R,NaN,NaN,NaN
2,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,L,NaN,NaN,NaN
3,4330000534,63280481,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-06-20,2,R,NaN,NaN,NaN
4,4330000534,60228678,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2020-06-01,1,NaN,INDEX-2,754.0,24.8
5,4330000534,60687102,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-06-10,1,NaN,INDEX-2,745.0,24.5
6,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,L,INDEX-1,387.0,12.7
7,4330000534,67674008,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-06-03,2,R,INDEX-1,387.0,12.7
8,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,L,INDEX,NaN,NaN
9,4330000534,451457076,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-06-25,2,R,INDEX,NaN,NaN


In [102]:
cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")

### **Criteria**:
#### 1. No pathology ever
* Already met, no need to check
#### 2. At <span style="color:#8A2BE2;">**INDEX**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 3. At <span style="color:#8A2BE2;">**INDEX**</span>, "Study" = "SCREEN" (Not availabe) 
#### 4. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Series" = "DBT" (Not availabe)
#### 5. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 6. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Study" = "SCREEN" (Not availabe)

In [103]:
normal_cats = ['1 - Negative', '2 - Benign finding']

recall_cats = [
    '0 - Need additional imaging evaluation',
    '3 - Probably benign - short interval follow-up',
    '4 - Suspicious abnormality, biopsy should be considered',
    '4A - Suspicious abnormality - biopsy should be considered - low suspicion',
    '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion',
    '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion',
    '5 - Highly suggestive of malignancy, appropriate action should be taken',
    '6 - Known biopsy proven malignancy'
]

In [104]:
# Patients with ANY recall finding at INDEX-1 (disqualify, takes priority)
recall_patients = set(cohort.loc[
    ((cohort['Index'] == 'INDEX-1') | (cohort['Index'] == 'INDEX')) &
    (cohort['FINDING_CATEGORY'].isin(recall_cats)),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX
normal_patients_at_index = set(cohort.loc[
    (cohort['Index'] == 'INDEX') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX-1
normal_patients_at_index1 = set(cohort.loc[
    (cohort['Index'] == 'INDEX-1') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)),
    'PATIENT_STUDY_ID'
].unique())

normal_patients = normal_patients_at_index & normal_patients_at_index1

# Exclude wins: remove any patient who appeared in recall
qualifying_patients = normal_patients - recall_patients

print(f"Normal patients     : {len(normal_patients)}")
print(f"Recall patients     : {len(recall_patients)}")
print(f"Overlap removed     : {len(normal_patients & recall_patients)}")
print(f"Qualifying patients : {len(qualifying_patients)}")

normal = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].copy()
print(f"normal rows      : {len(normal)}")
print(f"Index breakdown:\n{normal['Index'].value_counts(dropna=False)}")

Normal patients     : 36084
Recall patients     : 0
Overlap removed     : 0
Qualifying patients : 36084
normal rows      : 401508
Index breakdown:
Index
NaN        209543
INDEX       71440
INDEX-1     70963
INDEX-2     46813
INDEX-3      2502
INDEX-4       204
INDEX-5        40
INDEX-6         3
Name: count, dtype: int64


In [106]:
normal_cohort = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].reset_index(drop=True)
normal_cohort.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True, inplace=True)

normal_cohort["EXAM_COMPLETED_DATE"] = normal_cohort["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")

##### <span style="color:#FF6347;">**SAVE**</span> **normal_cohort**

In [107]:
normal_cohort['PATIENT_STUDY_ID'].nunique()

36084

In [ ]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'normal_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'normal_candidate_cohort' + ".xlsx")

normal_cohort.to_excel(output_file, index=False)

##### <span style="color:#FF6347;">**SAVE**</span> **normal** (only retain INDX & INDEX-1)

In [62]:
normal= normal_cohort[(normal_cohort["Index"]=="INDEX") | (normal_cohort["Index"]=="INDEX-1")]

In [63]:
normal['PATIENT_STUDY_ID'].nunique()

0

In [ ]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'normal' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'normal_candidate' + ".xlsx")

normal.to_excel(output_file, index=False)